In [ ]:
import torch
import torch.nn.functional as F
import sys
import os
sys.path.append("../")       # one folder up
sys.path.append("../models")

from models.baseline_Unet_ViT import (
    Convblock,
    EncodeBlock,
    DecodeBlock,
    ViTBottleneck,
    BottleneckViT,
    U_net_ViT,
)


def print_header(msg):
    print("\n" + "=" * 80)
    print(msg)
    print("=" * 80)


In [ ]:
#test convblock
def test_convblock():
    print_header("TEST 1: Convblock")

    x = torch.randn(1, 3, 64, 64)
    block = Convblock(3, 16, filter_size=3, padding=1, normalize=True)

    out = block(x)
    print("Convblock output shape:", out.shape)

#test encode block
def test_encodeblock():
    print_header("TEST 2: EncodeBlock")

    x = torch.randn(1, 3, 128, 128)
    block = EncodeBlock(3, 16, residual_channels=16, padding=1, normalize=True)

    pooled, residual = block(x)
    print("EncodeBlock pooled shape:  ", pooled.shape)
    print("EncodeBlock residual shape:", residual.shape)

#test decode block
def test_decodeblock():
    print_header("TEST 3: DecodeBlock")

    # Simulate decoder stage input
    x = torch.randn(1, 128, 32, 32)

    residual = torch.randn(1, 64, 64, 64)


    block = DecodeBlock(128, 64, up_size=2, padding=1, residual="interpolate")

    out = block(x, residual)
    print("DecodeBlock output shape:", out.shape)

#test ViT bottleneck only
def test_vitbottleneck():
    print_header("TEST 4: ViTBottleneck (standalone)")

    x = torch.randn(1, 128, 16, 16)  # b, c, h, w (typical bottleneck)
    vit = ViTBottleneck(128, num_layers=2, num_heads=4)

    out = vit(x)
    print("ViTBottleneck output shape:", out.shape)

#test BottleneckViT (conv + transformer)
def test_bottleneck_vit():
    print_header("TEST 5: BottleneckViT (conv + transformer)")

    x = torch.randn(1, 64, 32, 32)
    bn = BottleneckViT(
        in_channels=64,
        bottleneck_channels=128,
        normalize=True,
        vit_num_layers=2,
        vit_num_heads=4,
        filter_size=3,
    )

    out = bn(x)
    print("BottleneckViT output shape:", out.shape)

#test full U-Net ViT on single size
def test_full_model_single():
    print_header("TEST 6: Full U_net_ViT model (single pass)")

    x = torch.randn(1, 1, 572, 572)

    model = U_net_ViT(
        encode_in=(1, 64, 128, 256),
        encode_out=(64, 128, 256, 512),
        decode_in=(1024, 512, 256, 128),
        decode_out=(512, 256, 128, 64),
        normalize=True,
        upsampling="bilinear",
        residual="interpolate",
        vit_num_layers=2,
        vit_num_heads=4,
    )

    out = model(x)
    print("Model output shape:", out.shape)

#test full U-Net ViT on multiple input sizes
def test_multi_input_sizes():
    print_header("TEST 7: Full model with multiple input sizes")

    sizes = [
        (128, 128),
        (256, 256),
        (320, 320),
        (572, 572),
        (1270, 1350),   # dataset size
    ]

    model = U_net_ViT()

    for h, w in sizes:
        x = torch.randn(1, 1, h, w)
        y = model(x)
        print(f"Input {h}x{w} → Output:", y.shape)

#test different transformer variants in bottleneck (multi layesrs/heads)
def test_transformer_variants():
    print_header("TEST 8: Transformer bottleneck variants")

    configs = [
        {"layers": 1, "heads": 2},
        {"layers": 2, "heads": 4},
        {"layers": 4, "heads": 8},
    ]

    x = torch.randn(1, 1, 256, 256)

    for cfg in configs:
        print(f"\nTesting ViT config: {cfg}")
        model = U_net_ViT(
            vit_num_layers=cfg["layers"],
            vit_num_heads=cfg["heads"],
        )
        out = model(x)
        print("Output:", out.shape)

#stress test with very small bottleneck
def test_small_bottleneck():
    print_header("TEST 9: Very small bottleneck (8x8) stress test")

    # Forcing small sizes to ensure flatten <-> spatial reshape is correct
    x = torch.randn(1, 1, 64, 64)

    model = U_net_ViT(
        encode_in=(1, 32),
        encode_out=(32, 64),
        decode_in=(128, 64),
        decode_out=(64, 32),
        vit_num_layers=3,
        vit_num_heads=4,
    )

    out = model(x)
    print("Output shape:", out.shape)

#patch sequence and size validation (token correctness test)
def test_token_shape_correctness():
    print_header("TEST 10: Token shape correctness in ViTBottleneck")

    b, c, h, w = 2, 64, 20, 20
    x = torch.randn(b, c, h, w)
    vit = ViTBottleneck(c)

    out = vit(x)
    print("Token test output:", out.shape)
    assert out.shape == (b, c, h, w), "Token reshape error!"


In [ ]:
if __name__ == "__main__":
    test_convblock()
    test_encodeblock()
    test_decodeblock()
    test_vitbottleneck()
    test_bottleneck_vit()
    test_full_model_single()
    test_multi_input_sizes()
    test_transformer_variants()
    test_small_bottleneck()
    test_token_shape_correctness()

    print("\nAll tests completed successfully.")

In [ ]:
#test specifically the different sizes:

if __name__ == "__main__":
    test_multi_input_sizes()

    print("\nAll tests completed successfully.")